# MSSQL — Demo Data Scripts

All scripts run on Azure SQL Server (SSMS / Azure Data Studio).

| Cell | Purpose | When to Run |
|------|---------|-------------|
| 1 | Initial Load (1.1M rows) | Before demo (one-time setup) |
| 2 | Incremental Changes | After first FULL migration (Section 5 of demo) |
| 3 | Demo Config | Copy JSON into migration_config.json |

> See `DEMO_PLAN.md` for the full recording guide.
> See `snowflake.ipynb` for Snowflake target DDL and verification queries.

## 1. Initial Load — Create Tables & Populate 1.1M Rows
Run on Azure SQL Server to create `TestDB` with:
- **Customers** — 100K rows (SCD2 target)
- **Products** — 5K rows (SCD2 target)
- **Orders** — 1M rows (SCD1 target)

In [ ]:
USE TestDB;
GO

-- ╔══════════════════════════════════════════════════════════════════════════════╗
-- ║  TABLE 1: CUSTOMERS (100,000 rows)                                          ║
-- ╚══════════════════════════════════════════════════════════════════════════════╝

IF OBJECT_ID('dbo.Customers', 'U') IS NOT NULL DROP TABLE dbo.Customers;
GO

CREATE TABLE dbo.Customers (
    CustomerID      INT IDENTITY(1,1) PRIMARY KEY,
    FirstName       VARCHAR(50)   NOT NULL,
    LastName        VARCHAR(50)   NOT NULL,
    Email           VARCHAR(100)  NOT NULL,
    Phone           VARCHAR(20),
    City            VARCHAR(50),
    State           VARCHAR(30),
    Country         VARCHAR(30)   DEFAULT 'USA',
    ZipCode         VARCHAR(10),
    CustomerType    VARCHAR(20),
    CreditLimit     DECIMAL(12,2),
    IsActive        BIT           DEFAULT 1,
    CreatedAt       DATETIME2     DEFAULT SYSUTCDATETIME(),
    UpdatedAt       DATETIME2     DEFAULT SYSUTCDATETIME()
);
GO

SET NOCOUNT ON;
DECLARE @i INT = 1;
DECLARE @batch INT = 10000;

WHILE @i <= 100000
BEGIN
    INSERT INTO dbo.Customers (FirstName, LastName, Email, Phone, City, State, Country, ZipCode, CustomerType, CreditLimit, IsActive, CreatedAt, UpdatedAt)
    SELECT TOP (@batch)
        LEFT('FName' + CAST(ROW_NUMBER() OVER(ORDER BY (SELECT NULL)) + @i - 1 AS VARCHAR), 50),
        LEFT('LName' + CAST(ROW_NUMBER() OVER(ORDER BY (SELECT NULL)) + @i - 1 AS VARCHAR), 50),
        'customer' + CAST(ROW_NUMBER() OVER(ORDER BY (SELECT NULL)) + @i - 1 AS VARCHAR) + '@company' + CAST(ABS(CHECKSUM(NEWID())) % 100 AS VARCHAR) + '.com',
        '+1-' + RIGHT('000' + CAST(ABS(CHECKSUM(NEWID())) % 999 AS VARCHAR), 3) + '-' + RIGHT('0000' + CAST(ABS(CHECKSUM(NEWID())) % 9999 AS VARCHAR), 4),
        CHOOSE(ABS(CHECKSUM(NEWID())) % 10 + 1, 'New York','Los Angeles','Chicago','Houston','Phoenix','Philadelphia','San Antonio','San Diego','Dallas','Austin'),
        CHOOSE(ABS(CHECKSUM(NEWID())) % 10 + 1, 'NY','CA','IL','TX','AZ','PA','TX','CA','TX','TX'),
        CHOOSE(ABS(CHECKSUM(NEWID())) % 5 + 1, 'USA','USA','USA','Canada','UK'),
        RIGHT('00000' + CAST(ABS(CHECKSUM(NEWID())) % 99999 AS VARCHAR), 5),
        CHOOSE(ABS(CHECKSUM(NEWID())) % 4 + 1, 'Enterprise','Business','Individual','Partner'),
        CAST(ABS(CHECKSUM(NEWID())) % 50000 + 1000 AS DECIMAL(12,2)),
        CASE WHEN ABS(CHECKSUM(NEWID())) % 100 < 90 THEN 1 ELSE 0 END,
        DATEADD(DAY, -ABS(CHECKSUM(NEWID())) % 730, SYSUTCDATETIME()),
        DATEADD(DAY, -ABS(CHECKSUM(NEWID())) % 30, SYSUTCDATETIME())
    FROM sys.all_objects a CROSS JOIN sys.all_objects b;
    SET @i = @i + @batch;
END;
GO

-- ╔══════════════════════════════════════════════════════════════════════════════╗
-- ║  TABLE 2: PRODUCTS (5,000 rows)                                             ║
-- ╚══════════════════════════════════════════════════════════════════════════════╝

IF OBJECT_ID('dbo.Products', 'U') IS NOT NULL DROP TABLE dbo.Products;
GO

CREATE TABLE dbo.Products (
    ProductID       INT IDENTITY(1,1) PRIMARY KEY,
    ProductName     VARCHAR(100)  NOT NULL,
    Category        VARCHAR(50),
    SubCategory     VARCHAR(50),
    Brand           VARCHAR(50),
    SKU             VARCHAR(30),
    UnitPrice       DECIMAL(10,2) NOT NULL,
    CostPrice       DECIMAL(10,2),
    Weight          DECIMAL(8,2),
    IsActive        BIT           DEFAULT 1,
    CreatedAt       DATETIME2     DEFAULT SYSUTCDATETIME(),
    UpdatedAt       DATETIME2     DEFAULT SYSUTCDATETIME()
);
GO

SET NOCOUNT ON;
DECLARE @p INT = 1;
WHILE @p <= 5000
BEGIN
    ;WITH nums AS (
        SELECT TOP (1000) ROW_NUMBER() OVER(ORDER BY (SELECT NULL)) AS n
        FROM sys.all_objects a CROSS JOIN sys.all_objects b
    )
    INSERT INTO dbo.Products (ProductName, Category, SubCategory, Brand, SKU, UnitPrice, CostPrice, Weight, IsActive, CreatedAt, UpdatedAt)
    SELECT
        ISNULL('Product-' + CAST(n + @p - 1 AS VARCHAR) + '-' + CHOOSE(ABS(CHECKSUM(NEWID()) % 2147483646) % 5 + 1, 'Pro','Elite','Basic','Ultra','Max'), 'Product-' + CAST(n + @p - 1 AS VARCHAR)),
        CHOOSE(ABS(CHECKSUM(NEWID()) % 2147483646) % 8 + 1, 'Electronics','Clothing','Home','Sports','Food','Beauty','Toys','Office'),
        CHOOSE(ABS(CHECKSUM(NEWID()) % 2147483646) % 6 + 1, 'Premium','Standard','Budget','Luxury','Eco','Value'),
        CHOOSE(ABS(CHECKSUM(NEWID()) % 2147483646) % 10 + 1, 'BrandA','BrandB','BrandC','BrandD','BrandE','BrandF','BrandG','BrandH','BrandI','BrandJ'),
        'SKU-' + RIGHT('00000' + CAST(n + @p - 1 AS VARCHAR), 6),
        CAST(ABS(CHECKSUM(NEWID()) % 2147483646) % 999 + 1 AS DECIMAL(10,2)) + 0.99,
        CAST(ABS(CHECKSUM(NEWID()) % 2147483646) % 500 + 1 AS DECIMAL(10,2)) + 0.50,
        CAST(ABS(CHECKSUM(NEWID()) % 2147483646) % 50 AS DECIMAL(8,2)) + 0.1,
        CASE WHEN ABS(CHECKSUM(NEWID()) % 2147483646) % 100 < 85 THEN 1 ELSE 0 END,
        DATEADD(DAY, -ABS(CHECKSUM(NEWID()) % 2147483646) % 365, SYSUTCDATETIME()),
        DATEADD(DAY, -ABS(CHECKSUM(NEWID()) % 2147483646) % 30, SYSUTCDATETIME())
    FROM nums;
    SET @p = @p + 1000;
END;
GO

-- ╔══════════════════════════════════════════════════════════════════════════════╗
-- ║  TABLE 3: ORDERS (1,000,000 rows)                                           ║
-- ╚══════════════════════════════════════════════════════════════════════════════╝

IF OBJECT_ID('dbo.Orders', 'U') IS NOT NULL DROP TABLE dbo.Orders;
GO

CREATE TABLE dbo.Orders (
    OrderID         INT IDENTITY(1,1) PRIMARY KEY,
    CustomerID      INT           NOT NULL,
    ProductID       INT           NOT NULL,
    OrderDate       DATE          NOT NULL,
    ShipDate        DATE,
    Quantity        INT           NOT NULL,
    UnitPrice       DECIMAL(10,2) NOT NULL,
    Discount        DECIMAL(5,2)  DEFAULT 0,
    TotalAmount     DECIMAL(12,2) NOT NULL,
    OrderStatus     VARCHAR(20),
    PaymentMethod   VARCHAR(20),
    ShippingMethod  VARCHAR(20),
    Region          VARCHAR(30),
    CreatedAt       DATETIME2     DEFAULT SYSUTCDATETIME(),
    UpdatedAt       DATETIME2     DEFAULT SYSUTCDATETIME()
);
GO

DECLARE @o INT = 1;
DECLARE @batchSize INT = 50000;
WHILE @o <= 1000000
BEGIN
    INSERT INTO dbo.Orders (CustomerID, ProductID, OrderDate, ShipDate, Quantity, UnitPrice, Discount, TotalAmount, OrderStatus, PaymentMethod, ShippingMethod, Region, CreatedAt, UpdatedAt)
    SELECT TOP (@batchSize)
        ABS(CHECKSUM(NEWID())) % 100000 + 1,
        ABS(CHECKSUM(NEWID())) % 5000 + 1,
        DATEADD(DAY, -ABS(CHECKSUM(NEWID())) % 730, CAST(GETDATE() AS DATE)),
        DATEADD(DAY, -ABS(CHECKSUM(NEWID())) % 720, CAST(GETDATE() AS DATE)),
        ABS(CHECKSUM(NEWID())) % 20 + 1,
        CAST(ABS(CHECKSUM(NEWID())) % 500 + 10 AS DECIMAL(10,2)),
        CAST(ABS(CHECKSUM(NEWID())) % 30 AS DECIMAL(5,2)),
        CAST((ABS(CHECKSUM(NEWID())) % 20 + 1) * (ABS(CHECKSUM(NEWID())) % 500 + 10) AS DECIMAL(12,2)),
        CHOOSE(ABS(CHECKSUM(NEWID())) % 5 + 1, 'Completed','Shipped','Processing','Cancelled','Returned'),
        CHOOSE(ABS(CHECKSUM(NEWID())) % 4 + 1, 'CreditCard','DebitCard','PayPal','BankTransfer'),
        CHOOSE(ABS(CHECKSUM(NEWID())) % 3 + 1, 'Standard','Express','Overnight'),
        CHOOSE(ABS(CHECKSUM(NEWID())) % 6 + 1, 'North','South','East','West','Central','International'),
        DATEADD(DAY, -ABS(CHECKSUM(NEWID())) % 730, SYSUTCDATETIME()),
        DATEADD(DAY, -ABS(CHECKSUM(NEWID())) % 30, SYSUTCDATETIME())
    FROM sys.all_objects a CROSS JOIN sys.all_objects b;
    SET @o = @o + @batchSize;
END;
GO

-- Indexes for BCP performance
CREATE INDEX IX_Customers_UpdatedAt ON dbo.Customers(UpdatedAt);
CREATE INDEX IX_Orders_UpdatedAt ON dbo.Orders(UpdatedAt);
CREATE INDEX IX_Orders_CustomerID ON dbo.Orders(CustomerID);
CREATE INDEX IX_Products_UpdatedAt ON dbo.Products(UpdatedAt);
GO

-- Verify counts
SELECT 'Customers' AS TableName, COUNT(*) AS [RowCount] FROM dbo.Customers
UNION ALL SELECT 'Products', COUNT(*) FROM dbo.Products
UNION ALL SELECT 'Orders', COUNT(*) FROM dbo.Orders;
GO

## 2. Incremental Changes (Run AFTER first FULL migration)
Simulates real-world changes to trigger:
- **SCD2 versioning** on Customers (50 updates + 10 inserts) and Products (10 price changes)
- **SCD1 upsert** on Orders (100 new rows)

In [ ]:
USE TestDB;
GO

-- ╔══════════════════════════════════════════════════════════════════════════════╗
-- ║  STEP 1: UPDATE 50 Customers (triggers SCD2 — expire old, insert new)       ║
-- ╚══════════════════════════════════════════════════════════════════════════════╝

UPDATE dbo.Customers SET
    Email = 'updated_' + CAST(CustomerID AS VARCHAR) + '@newdomain.com',
    Phone = '999-' + RIGHT('000000' + CAST(CustomerID AS VARCHAR), 7),
    City = 'UpdatedCity-' + CAST(CustomerID AS VARCHAR),
    UpdatedAt = SYSUTCDATETIME()
WHERE CustomerID IN (
    1, 2, 3, 4, 5, 10, 15, 20, 25, 30,
    50, 75, 100, 200, 300, 400, 500, 600, 700, 800,
    1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 5500,
    6000, 7000, 8000, 9000, 10000, 15000, 20000, 25000, 30000, 35000,
    40000, 50000, 60000, 70000, 80000, 85000, 90000, 92000, 95000, 99000
);
PRINT 'Updated 50 customers (SCD2 will version these)';
GO

-- ╔══════════════════════════════════════════════════════════════════════════════╗
-- ║  STEP 2: INSERT 10 new Customers                                            ║
-- ╚══════════════════════════════════════════════════════════════════════════════╝

INSERT INTO dbo.Customers (FirstName, LastName, Email, Phone, City, State, Country, ZipCode, CustomerType, CreditLimit, IsActive, CreatedAt, UpdatedAt)
VALUES
    ('Demo', 'Alpha',   'demo.alpha@test.com',   '555-1001', 'New York',   'NY', 'USA',      '10001',  'Premium',  15000.00, 1, SYSUTCDATETIME(), SYSUTCDATETIME()),
    ('Demo', 'Beta',    'demo.beta@test.com',    '555-1002', 'London',     NULL, 'UK',       'EC1A',   'Standard', 5000.00,  1, SYSUTCDATETIME(), SYSUTCDATETIME()),
    ('Demo', 'Gamma',   'demo.gamma@test.com',   '555-1003', 'Mumbai',     'MH', 'India',    '400001', 'Premium',  12000.00, 1, SYSUTCDATETIME(), SYSUTCDATETIME()),
    ('Demo', 'Delta',   'demo.delta@test.com',   '555-1004', 'Sydney',     'NSW','Australia', '2000',  'Standard', 8000.00,  1, SYSUTCDATETIME(), SYSUTCDATETIME()),
    ('Demo', 'Epsilon', 'demo.epsilon@test.com', '555-1005', 'Toronto',    'ON', 'Canada',   'M5V',    'Premium',  20000.00, 1, SYSUTCDATETIME(), SYSUTCDATETIME()),
    ('Demo', 'Zeta',    'demo.zeta@test.com',    '555-1006', 'Berlin',     NULL, 'Germany',  '10115',  'Standard', 6000.00,  1, SYSUTCDATETIME(), SYSUTCDATETIME()),
    ('Demo', 'Eta',     'demo.eta@test.com',     '555-1007', 'Tokyo',      NULL, 'Japan',    '100',    'Premium',  25000.00, 1, SYSUTCDATETIME(), SYSUTCDATETIME()),
    ('Demo', 'Theta',   'demo.theta@test.com',   '555-1008', 'Paris',      NULL, 'France',   '75001',  'Standard', 7000.00,  1, SYSUTCDATETIME(), SYSUTCDATETIME()),
    ('Demo', 'Iota',    'demo.iota@test.com',    '555-1009', 'Dubai',      NULL, 'UAE',      '00000',  'Premium',  30000.00, 1, SYSUTCDATETIME(), SYSUTCDATETIME()),
    ('Demo', 'Kappa',   'demo.kappa@test.com',   '555-1010', 'Singapore',  NULL, 'Singapore','048',    'Standard', 9000.00,  1, SYSUTCDATETIME(), SYSUTCDATETIME());
PRINT 'Inserted 10 new customers';
GO

-- ╔══════════════════════════════════════════════════════════════════════════════╗
-- ║  STEP 3: UPDATE 10 Products (price increase → SCD2 versioning)              ║
-- ╚══════════════════════════════════════════════════════════════════════════════╝

UPDATE dbo.Products SET
    UnitPrice = UnitPrice * 1.10,
    UpdatedAt = SYSUTCDATETIME()
WHERE ProductID IN (1, 2, 3, 4, 5, 10, 20, 50, 100, 500);
PRINT 'Updated 10 products (10% price increase → SCD2 preserves old prices)';
GO

-- ╔══════════════════════════════════════════════════════════════════════════════╗
-- ║  STEP 4: INSERT 100 new Orders (SCD1 — simple upsert)                       ║
-- ╚══════════════════════════════════════════════════════════════════════════════╝

;WITH nums AS (
    SELECT TOP (100) ROW_NUMBER() OVER(ORDER BY (SELECT NULL)) AS n
    FROM sys.all_objects
)
INSERT INTO dbo.Orders (CustomerID, ProductID, OrderDate, ShipDate, Quantity, UnitPrice, Discount, TotalAmount, OrderStatus, PaymentMethod, ShippingMethod, Region, CreatedAt, UpdatedAt)
SELECT
    ABS(CHECKSUM(NEWID())) % 100000 + 1,
    ABS(CHECKSUM(NEWID())) % 5000 + 1,
    CAST(SYSUTCDATETIME() AS DATE),
    DATEADD(DAY, ABS(CHECKSUM(NEWID())) % 7 + 1, CAST(SYSUTCDATETIME() AS DATE)),
    ABS(CHECKSUM(NEWID())) % 10 + 1,
    CAST(ABS(CHECKSUM(NEWID())) % 500 + 10 AS DECIMAL(10,2)),
    CAST(ABS(CHECKSUM(NEWID())) % 20 AS DECIMAL(5,2)),
    CAST(ABS(CHECKSUM(NEWID())) % 5000 + 50 AS DECIMAL(12,2)),
    CHOOSE(ABS(CHECKSUM(NEWID())) % 4 + 1, 'Pending','Confirmed','Shipped','Processing'),
    CHOOSE(ABS(CHECKSUM(NEWID())) % 4 + 1, 'CreditCard','DebitCard','PayPal','BankTransfer'),
    CHOOSE(ABS(CHECKSUM(NEWID())) % 3 + 1, 'Standard','Express','Overnight'),
    CHOOSE(ABS(CHECKSUM(NEWID())) % 6 + 1, 'North','South','East','West','Central','International'),
    SYSUTCDATETIME(),
    SYSUTCDATETIME()
FROM nums;
PRINT 'Inserted 100 new orders';
GO

-- ╔══════════════════════════════════════════════════════════════════════════════╗
-- ║  VERIFICATION                                                               ║
-- ╚══════════════════════════════════════════════════════════════════════════════╝

SELECT 'Customers (Changed)' AS Metric, COUNT(*) AS Cnt FROM dbo.Customers WHERE UpdatedAt >= DATEADD(MINUTE, -5, SYSUTCDATETIME())
UNION ALL SELECT 'Products (Changed)', COUNT(*) FROM dbo.Products WHERE UpdatedAt >= DATEADD(MINUTE, -5, SYSUTCDATETIME())
UNION ALL SELECT 'Orders (New)', COUNT(*) FROM dbo.Orders WHERE CreatedAt >= DATEADD(MINUTE, -5, SYSUTCDATETIME());
GO

PRINT '─────────────────────────────────────────────';
PRINT 'READY FOR INCREMENTAL MIGRATION:';
PRINT '  Customers: 60 rows (50 updated + 10 new) → SCD2';
PRINT '  Products:  10 rows (price updates) → SCD2';
PRINT '  Orders:    100 rows (new inserts) → SCD1';
PRINT '─────────────────────────────────────────────';
GO

## 3. Demo Config (migration_config.json)
Run this cell to generate the config JSON. Copy into `migration_config.json`.

In [ ]:
import json

demo_config = {
    "export_dir": "./export",
    "tables": [
        {
            "source_db": "TestDB",
            "source_schema": "dbo",
            "source_table": "Customers",
            "target_db": "ANALYTICS",
            "target_schema": "PUBLIC",
            "target_table": "CUSTOMERS",
            "primary_key": "CUSTOMERID",
            "load_type": "incremental",
            "cdc_columns": "UPDATEDAT",
            "cdc_type": "TIMESTAMP",
            "scd_type": 2,
            "execution_mode": "FULL",
            "delimiter": "|",
            "cloud_path": "https://tapocblob.blob.core.windows.net/mssql-sf/",
            "warehouse_name": "COMPUTE_WH",
            "active": True
        },
        {
            "source_db": "TestDB",
            "source_schema": "dbo",
            "source_table": "Products",
            "target_db": "ANALYTICS",
            "target_schema": "PUBLIC",
            "target_table": "PRODUCTS",
            "primary_key": "PRODUCTID",
            "load_type": "incremental",
            "cdc_columns": "UPDATEDAT",
            "cdc_type": "TIMESTAMP",
            "scd_type": 2,
            "execution_mode": "FULL",
            "delimiter": "|",
            "cloud_path": "https://tapocblob.blob.core.windows.net/mssql-sf/",
            "warehouse_name": "COMPUTE_WH",
            "active": True
        },
        {
            "source_db": "TestDB",
            "source_schema": "dbo",
            "source_table": "Orders",
            "target_db": "ANALYTICS",
            "target_schema": "PUBLIC",
            "target_table": "ORDERS",
            "primary_key": "ORDERID",
            "load_type": "incremental",
            "cdc_columns": "UPDATEDAT",
            "cdc_type": "TIMESTAMP",
            "scd_type": 1,
            "execution_mode": "FULL",
            "delimiter": "|",
            "cloud_path": "https://tapocblob.blob.core.windows.net/mssql-sf/",
            "warehouse_name": "COMPUTE_WH",
            "active": True
        }
    ]
}

print(json.dumps(demo_config, indent=2))
print("\n--- Copy above JSON into migration_config.json ---")